# 📚 Section 12: Missing Value Handling
> 결측치 처리 — Detect, choose a strategy (drop/impute/interpolate/conditional), visualize, and verify — the final section of this NumPy series

---
# 🎯 Learning Objective
Today I want to learn:
- [x] How to detect missing values (count, location, ratio, co-occurrence pattern) with `np.isnan()`/`np.isfinite()`
- [x] The four missing-value handling strategies — drop, statistical imputation, interpolation, conditional replacement — and when each is the right choice
- [x] How to build a text-based missing-value report and a full detect→clean→verify pipeline function

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

Missing value handling is the discipline of finding gaps in data (`np.nan`) and deciding, deliberately, what to do about each one. The original material opens with a blunt fact: roughly 80% of real-world data contains missing values, and the strategy you pick can change your analysis results completely — which is why the workflow always follows the same order: detect → understand why → choose a strategy → verify.

결측치 처리는 데이터의 빈 곳(`np.nan`)을 찾아내고, 각각에 대해 무엇을 할지 의도적으로 결정하는 작업입니다. 원문은 실제 데이터의 약 80%가 결측치를 포함한다는 직설적인 사실로 시작하며, 어떤 전략을 선택하느냐가 분석 결과를 완전히 바꿀 수 있다고 말합니다 — 그래서 워크플로우는 항상 같은 순서를 따릅니다: 탐지 → 원인 파악 → 전략 선택 → 검증.

### Four strategies, and when to reach for each / 4가지 전략과 사용 시점

| Strategy / 전략 | Best when / 적합한 상황 | Trade-off / 트레이드오프 |
|---|---|---|
| Drop (제거) | <5% missing, random, plenty of rows (결측 5% 미만, 무작위, 데이터 충분) | Simple, but loses data (간단하지만 데이터 손실) |
| Imputation (통계값 대체) | 5-20% missing (결측 5~20%) | Preserves rows, but shrinks variance (행은 유지, 분산은 축소) |
| Interpolation (보간) | Ordered / time-series data (순서가 있는 시계열 데이터) | Great for trends, meaningless for unordered IDs (추세엔 좋지만 순서 없는 데이터엔 무의미) |
| Conditional (조건부 대체) | Missingness has a known business reason (결측 원인이 명확할 때) | Most accurate, most manual (가장 정확하지만 수작업 많음) |

## Why do we use it?
*(When is it useful?)*

Because silently ignoring missing values (or handling them carelessly) doesn't make them disappear — it just hides bad assumptions inside a mean, a sum, or a model until they surface later as a wrong business decision.

결측치를 조용히 무시하거나(또는 부주의하게 처리) 없앤다고 사라지는 게 아닙니다 — 평균, 합계, 모델 안에 나쁜 가정을 숨겨둘 뿐이고, 이는 나중에 잘못된 비즈니스 의사결정으로 드러납니다.

## When is it used in Business Analytics?
*(Real-world use case)*

- Running a data-quality check immediately after loading any CSV or database export.
  CSV나 DB에서 데이터를 불러온 직후 데이터 품질 점검을 할 때.
- Deciding whether to drop a handful of bad rows or a whole unreliable column.
  소수의 문제 행을 버릴지, 신뢰할 수 없는 컬럼 전체를 버릴지 결정할 때.
- Filling gaps in a daily/monthly time series (DAU, revenue) so trend charts aren't broken.
  일별/월별 시계열(DAU, 매출)의 빈 곳을 채워서 추세 차트가 끊기지 않게 할 때.
- Building one reusable pipeline function the team can run on every new data drop.
  팀이 매번 새 데이터에 실행할 수 있는 하나의 재사용 파이프라인 함수를 만들 때.

---
# 📝 Syntax

## Basic Syntax

In [1]:
import numpy as np

data = np.array([1200.0, np.nan, 1100.0, np.nan, 1600.0])

# detect
print(np.isnan(data))
print(np.sum(np.isnan(data)))

# drop
clean = data[~np.isnan(data)]
print(clean)

# impute with the mean of valid values
filled = np.where(np.isnan(data), np.nanmean(data), data)
print(filled)

[False  True False  True False]
2
[1200. 1100. 1600.]
[1200. 1300. 1100. 1300. 1600.]


## Common Variations

In [2]:
import numpy as np

data = np.array([1200.0, np.nan, 1100.0, np.nan, 1600.0])

# median is often safer than mean when outliers are possible
filled_median = np.where(np.isnan(data), np.nanmedian(data), data)
print(filled_median)

# linear interpolation for ordered/time-series data
days_known = np.array([0, 2, 4])
sales_known = np.array([100, 120, 140])
sales_interp = np.interp(np.arange(5), days_known, sales_known)
print(sales_interp)

# 2D: missing ratio per column (axis=0) vs per row (axis=1)
matrix = np.array([[1.0, np.nan], [2.0, 3.0], [np.nan, np.nan]])
print(np.isnan(matrix).mean(axis=0))   # per-column ratio
print(np.isnan(matrix).mean(axis=1))   # per-row ratio

[1200. 1200. 1100. 1200. 1600.]
[100. 110. 120. 130. 140.]
[0.33333333 0.66666667]
[0.5 0.  1. ]


---
# 🧪 Small Examples

## Example 1: Detecting Missing Values — Count, Location, Pattern
### 12-1. 결측치 탐지 — 개수·위치·비율 파악

In [3]:
import numpy as np

# Business: 2D data, missing spread across columns
data = np.array([
    [1200, np.nan, 3.8],
    [1350, 3500, np.nan],
    [np.nan, 2800, 3.9],
    [1400, 3700, 3.8],
    [1600, np.nan, np.nan],
])

nan_by_col = np.sum(np.isnan(data), axis=0)
nan_by_row = np.sum(np.isnan(data), axis=1)
nan_pct_col = nan_by_col / data.shape[0] * 100
total_nan = np.sum(np.isnan(data))
total_cells = data.size

print("NaN count per column:", nan_by_col)
print("NaN %      per column:", np.round(nan_pct_col, 1))
print("NaN count per row:", nan_by_row)
print(f"\nOverall missing rate: {total_nan}/{total_cells} "
      f"({total_nan/total_cells*100:.1f}%)")

# which columns tend to be missing together?
nan_mask = np.isnan(data)
cols = ["revenue", "visitors", "cvr"]
print("\nCo-missing pattern:")
for i in range(3):
    for j in range(i+1, 3):
        both = np.sum(nan_mask[:, i] & nan_mask[:, j])
        if both > 0:
            print(f"  {cols[i]} + {cols[j]} missing together: {both} row(s)")

NaN count per column: [1 2 2]
NaN %      per column: [20. 40. 40.]
NaN count per row: [1 1 1 0 2]

Overall missing rate: 5/15 (33.3%)

Co-missing pattern:
  visitors + cvr missing together: 1 row(s)


## Example 2: Dropping Rows — When Missing Data Is Rare
### 12-2. 결측치 처리 전략 1 — 제거 (Drop)

In [4]:
import numpy as np

data = np.array([
    [1200, 3200, 3.8],
    [1350, np.nan, 3.9],
    [1100, 2800, 3.9],
    [np.nan, 3700, 3.8],
    [1600, 4100, np.nan],
])

# keep only rows where every column is valid
valid_rows = ~np.any(np.isnan(data), axis=1)
clean = data[valid_rows]

print("Original shape:", data.shape)
print("After dropping:", clean.shape)
print(f"Rows removed: {data.shape[0] - clean.shape[0]}")
print("Clean data:\n", clean)

# a softer version: keep rows with at least 2 valid values
valid_count = np.sum(~np.isnan(data), axis=1)
kept_thresh = data[valid_count >= 2]
print(f"\nWith a 'keep if >=2 valid values' rule: {kept_thresh.shape[0]} rows kept")

Original shape: (5, 3)
After dropping: (2, 3)
Rows removed: 3
Clean data:
 [[1200.  3200.     3.8]
 [1100.  2800.     3.9]]

With a 'keep if >=2 valid values' rule: 5 rows kept


## Example 3: Statistical Imputation — Mean, Median, Mode, and Group-wise
### 12-3. 결측치 처리 전략 2 — 통계값 대체 (Imputation)

In [5]:
import numpy as np
import pandas as pd

data = np.array([1200.0, np.nan, 1100.0, 1400.0, np.nan,
                  1500.0, np.nan, 1250.0, 1450.0, 1700.0])

# mean and median imputation
mean_val = np.nanmean(data)
median_val = np.nanmedian(data)
imputed_mean = np.where(np.isnan(data), mean_val, data)
imputed_median = np.where(np.isnan(data), median_val, data)
print(f"Mean ({mean_val:.0f}) imputed:", imputed_mean)
print(f"Median ({median_val:.0f}) imputed:", imputed_median)

# mode imputation for a categorical (string) array
channels = np.array(["search", None, "sns", "search", None, "email", "search", None])
valid_ch = channels[channels != None]
vals, counts = np.unique(valid_ch, return_counts=True)
mode_val = vals[np.argmax(counts)]
imputed_ch = np.where(channels == None, mode_val, channels)
print(f"\nMode: {mode_val}")
print("Mode-imputed channels:", imputed_ch)

# group-wise mean is usually more accurate than one global mean
df = pd.DataFrame({
    "channel": ["search", "search", "sns", "sns", "search", "sns"],
    "revenue": [1200.0, np.nan, 800.0, np.nan, 1350.0, 850.0],
})
group_means = df.groupby("channel")["revenue"].transform("mean")
df["revenue_filled"] = df["revenue"].fillna(group_means)
print("\nGroup-wise mean imputation:\n", df)

Mean (1371) imputed: [1200.         1371.42857143 1100.         1400.         1371.42857143
 1500.         1371.42857143 1250.         1450.         1700.        ]
Median (1400) imputed: [1200. 1400. 1100. 1400. 1400. 1500. 1400. 1250. 1450. 1700.]

Mode: search
Mode-imputed channels: ['search' 'search' 'sns' 'search' 'search' 'email' 'search' 'search']

Group-wise mean imputation:
   channel  revenue  revenue_filled
0  search   1200.0          1200.0
1  search      NaN          1275.0
2     sns    800.0           800.0
3     sns      NaN           825.0
4  search   1350.0          1350.0
5     sns    850.0           850.0


## Example 4: Interpolation — Filling Gaps in a Time Series
### 12-4. 결측치 처리 전략 3 — 보간 (Interpolation)

In [6]:
import numpy as np
import pandas as pd

# NumPy: manual linear interpolation with np.interp
days_known = np.array([0, 2, 5, 8, 10])
sales_known = np.array([100, 120, 115, 140, 155])
days_all = np.arange(11)
sales_interp = np.interp(days_all, days_known, sales_known)
print("np.interp result:", np.round(sales_interp, 1))

# Pandas: several interpolation strategies side by side
monthly = pd.Series(
    [1200, np.nan, 1100, np.nan, np.nan, 1500, 1300, np.nan, 1450, 1700, np.nan, 2000],
    index=pd.date_range("2024-01", periods=12, freq="MS")
)

linear = monthly.interpolate(method="linear")
spline = monthly.interpolate(method="spline", order=2)
forward_fill = monthly.ffill()   # modern syntax -- fillna(method='ffill') is removed
back_fill = monthly.bfill()      # in recent pandas versions

print("\nOriginal:      ", monthly.to_numpy())
print("Linear interp: ", linear.round(0).to_numpy())
print("Spline interp: ", spline.round(0).to_numpy())
print("Forward fill:  ", forward_fill.to_numpy())
print("Backward fill: ", back_fill.to_numpy())

np.interp result: [100.  110.  120.  118.3 116.7 115.  123.3 131.7 140.  147.5 155. ]

Original:       [1200.   nan 1100.   nan   nan 1500. 1300.   nan 1450. 1700.   nan 2000.]
Linear interp:  [1200. 1150. 1100. 1233. 1367. 1500. 1300. 1375. 1450. 1700. 1850. 2000.]
Spline interp:  [1200. 1083. 1100. 1254. 1487. 1500. 1300. 1283. 1450. 1700. 1896. 2000.]
Forward fill:   [1200. 1200. 1100. 1100. 1100. 1500. 1300. 1300. 1450. 1700. 1700. 2000.]
Backward fill:  [1200. 1100. 1100. 1500. 1500. 1500. 1300. 1450. 1450. 1700. 2000. 2000.]


## Example 5: Conditional Replacement — Business-Logic-Driven Fills
### 12-5. 결측치 처리 전략 4 — 조건부 대체 (np.where 활용)

In [7]:
import numpy as np

revenue = np.array([1200.0, np.nan, 0.0, np.nan, 1600.0, np.nan])
visitors = np.array([3200, 0, 2800, 3700, 4100, 3900])
# rule: visitors==0 -> revenue=0 (no visits, no revenue)
#       NaN with visitors>0 -> revenue = mean of valid (nonzero) revenue
#       otherwise -> keep as is

valid_mean = np.nanmean(revenue[revenue > 0])

result = np.where(
    visitors == 0,             # condition 1
    0.0,                       # no visitors -> revenue is 0
    np.where(
        np.isnan(revenue),     # condition 2
        valid_mean,            # has visitors but NaN -> use the mean
        revenue                # condition 3: keep original
    )
)

print("Original revenue:", revenue)
print("Visitors:        ", visitors)
print("Conditional fill:", result)
print(f"(mean used = {valid_mean:.0f})")

Original revenue: [1200.   nan    0.   nan 1600.   nan]
Visitors:         [3200    0 2800 3700 4100 3900]
Conditional fill: [1200.    0.    0. 1400. 1600. 1400.]
(mean used = 1400)


## Example 6: A Text-Based Missing-Value Report
### 12-6. 결측치 시각화 리포트 (텍스트 기반)

In [8]:
import numpy as np
import pandas as pd

def missing_value_report(df, max_bar_width=30):
    """Text-based missing-value report -- no Matplotlib needed."""
    n = len(df)
    print(f"Rows: {n:,}  |  Columns: {len(df.columns)}")
    print(f"{'Column':>12} {'Missing':>8} {'Pct':>7}  Distribution")
    print("-" * 62)

    for col in df.columns:
        n_nan = df[col].isna().sum()
        pct = n_nan / n
        bar_len = int(pct * max_bar_width)
        bar = "#" * bar_len + "." * (max_bar_width - bar_len)
        flag = "[!!]" if pct > 0.20 else "[! ]" if pct > 0.05 else "[ok]"
        print(f"{flag} {col:>10} {n_nan:>6}  {pct*100:>5.1f}%  [{bar}]")

    total_nan = df.isna().sum().sum()
    total_cells = df.size
    print("-" * 62)
    print(f"Total missing cells: {total_nan:,} / {total_cells:,} "
          f"({total_nan/total_cells*100:.1f}%)")

# Business: quick report on a demo dataset
np.random.seed(0)
n = 300
df_demo = pd.DataFrame({
    "revenue": np.where(np.random.rand(n) < 0.03, np.nan,
                         np.random.randint(10000, 500000, n).astype(float)),
    "ad_spend": np.where(np.random.rand(n) < 0.07, np.nan,
                          np.random.randint(5000, 100000, n).astype(float)),
    "visitors": np.where(np.random.rand(n) < 0.12, np.nan,
                          np.random.randint(100, 5000, n).astype(float)),
    "cvr": np.where(np.random.rand(n) < 0.22, np.nan,
                     np.random.uniform(0.01, 0.1, n)),
})
missing_value_report(df_demo)

Rows: 300  |  Columns: 4
      Column  Missing     Pct  Distribution
--------------------------------------------------------------
[ok]    revenue     12    4.0%  [#.............................]
[! ]   ad_spend     23    7.7%  [##............................]
[! ]   visitors     41   13.7%  [####..........................]
[!!]        cvr     71   23.7%  [#######.......................]
--------------------------------------------------------------
Total missing cells: 147 / 1,200 (12.2%)


## Example 7: Practice — One Mini-Exercise per Strategy
### 연습 문제 (Practice)

**Note / 참고:** only 12-1 has a formal practice problem in the original material (used directly below as #1); 12-2 through 12-6 are self-designed to match this section's spirit.  
원문에서는 12-1만 정식 연습 문제가 있습니다(아래 #1로 그대로 사용) — 12-2~12-6은 이 섹션의 취지에 맞춰 직접 구성했습니다.

1. **(12-1, from the original)** Write a function that computes NaN count, NaN ratio, and the valid-value mean for one array, all at once. Test it on `[1.0, nan, 3.0, nan, 5.0, 6.0, nan, 8.0]`.  
   NaN 개수, NaN 비율, 유효값 평균을 한 번에 계산하는 함수를 작성하고 `[1.0, nan, 3.0, nan, 5.0, 6.0, nan, 8.0]`로 테스트하세요.  
2. **(12-2)** From a 2D array with scattered NaN, keep only rows with fewer than 2 missing values.  
   NaN이 흩어져 있는 2D 배열에서 결측이 2개 미만인 행만 남기세요.   
3. **(12-3)** Impute `[5, nan, 7, nan, 9, 11]` with both mean and median, print both.  
   `[5, nan, 7, nan, 9, 11]`을 평균과 중앙값으로 각각 대체하고 둘 다 출력하세요.
4. **(12-4)** Use `np.interp` to fill `[10, nan, nan, 40, 50]`, treating the array index as the x-position.  
   배열 인덱스를 x좌표로 취급해서 `np.interp`로 `[10, nan, nan, 40, 50]`을 채우세요.
5. **(12-5)** Given `price=[100, nan, 0, nan, 150]` and `in_stock=[1,1,0,1,1]`, set `price=0` where `in_stock=0`, else fill `NaN` with the mean of valid nonzero prices.  
   `price=[100, nan, 0, nan, 150]`, `in_stock=[1,1,0,1,1]`에서 `in_stock=0`이면 `price=0`, 아니면 `NaN`을 유효한 0 아닌 가격의 평균으로 채우세요.
6. **(12-6)** Print a one-line missing-ratio bar (using `#` and `.`) for a column with 30 missing out of 200 rows.  
   200행 중 30개가 결측인 컬럼에 대해 한 줄짜리 결측 비율 막대(`#`과 `.` 사용)를 출력하세요.

Fill in each `________` below, then run the cell.  
아래 각 `________`를 채운 후 셀을 실행하세요.

In [ ]:
# ✏️ Practice — replace each ________ line below, then run this cell.
# ✏️ 연습 문제 — 아래 각 ________ 줄을 채운 후 셀을 실행하세요.

import numpy as np

print("--- 1: nan_summary() ---")
def nan_summary(arr):
    arr = np.array(arr, dtype=float)
    n_nan = int(np.sum(np.isnan(arr)))
    ratio = n_nan / arr.size
    valid_mean = float(np.nanmean(arr))
    return {"nan_count": n_nan, "nan_ratio": ratio, "valid_mean": valid_mean}

test = np.array([1.0, np.nan, 3.0, np.nan, 5.0, 6.0, np.nan, 8.0])
s = nan_summary(test)
print(f"  nan_count={s['nan_count']}  nan_ratio={s['nan_ratio']:.1%}  valid_mean={s['valid_mean']:.1f}")
print("  -> never use np.mean() here: one NaN would make the whole mean NaN")

print("\n--- 2: drop by threshold ---")
data = np.array([
    [1.0, np.nan, 3.0],
    [4.0, 5.0, np.nan],
    [np.nan, np.nan, 9.0],
    [7.0, 8.0, 9.0],
    [np.nan, 2.0, np.nan],
])
n_missing = np.sum(np.isnan(data), axis=1)
kept = data[n_missing < 2]
print("  missing per row:", n_missing)
print(f"  kept {kept.shape[0]}/{data.shape[0]} rows (fewer than 2 NaNs):\n{kept}")

print("\n--- 3: mean/median impute ---")
vals = np.array([5.0, np.nan, 7.0, np.nan, 9.0, 11.0])
print(f"  mean={np.nanmean(vals):.1f}  median={np.nanmedian(vals):.1f}")
print("  mean-imputed  :", np.where(np.isnan(vals), np.nanmean(vals), vals))
print("  median-imputed:", np.where(np.isnan(vals), np.nanmedian(vals), vals))
print("  -> this sample has no outlier, so mean and median coincide")

print("\n--- 4: np.interp ---")
series = np.array([10.0, np.nan, np.nan, 40.0, 50.0])
x = np.arange(len(series))
known = ~np.isnan(series)
filled = np.interp(x, x[known], series[known])
print("  known x:", x[known], " known y:", series[known])
print("  interpolated:", filled)
print("  -> 10 to 40 over 3 steps: +10 per index -> 10, 20, 30, 40, 50")

print("\n--- 5: conditional fill ---")
price = np.array([100.0, np.nan, 0.0, np.nan, 150.0])
in_stock = np.array([1, 1, 0, 1, 1])
valid_mean = np.nanmean(price[price > 0])
filled_price = np.where(
    in_stock == 0,
    0.0,
    np.where(np.isnan(price), valid_mean, price),
)
print(f"  valid nonzero mean={valid_mean:.0f}")
print("  original:", price)
print("  filled  :", filled_price)
print("  -> in_stock==0 is checked FIRST so a missing+out-of-stock item becomes 0, not the mean")

print("\n--- 6: mini bar report ---")
n_missing, n_rows, width = 30, 200, 30
pct = n_missing / n_rows
bar_len = int(pct * width)
bar = "#" * bar_len + "." * (width - bar_len)
flag = "[!!]" if pct > 0.20 else "[! ]" if pct > 0.05 else "[ok]"
print(f"  {flag}  {n_missing}/{n_rows}  {pct*100:.1f}%  [{bar}]")
print("  -> 15% is in the 5-20% band: impute, don't drop the column")

print("\n✅ Fill in each ________ above with real code, then re-run to see all 6 results.")
print("✅ 위 각 ________ 를 실제 코드로 채운 뒤 다시 실행하면 6개 결과를 모두 볼 수 있습니다.")

--- 1: nan_summary() ---
  nan_count=3  nan_ratio=37.5%  valid_mean=4.6
  -> never use np.mean() here: one NaN would make the whole mean NaN

--- 2: drop by threshold ---
  missing per row: [1 1 2 0 2]
  kept 3/5 rows (fewer than 2 NaNs):
[[ 1. nan  3.]
 [ 4.  5. nan]
 [ 7.  8.  9.]]

--- 3: mean/median impute ---
  mean=8.0  median=8.0
  mean-imputed  : [ 5.  8.  7.  8.  9. 11.]
  median-imputed: [ 5.  8.  7.  8.  9. 11.]
  -> this sample has no outlier, so mean and median coincide

--- 4: np.interp ---
  known x: [0 3 4]  known y: [10. 40. 50.]
  interpolated: [10. 20. 30. 40. 50.]
  -> 10 to 40 over 3 steps: +10 per index -> 10, 20, 30, 40, 50

--- 5: conditional fill ---
  valid nonzero mean=125
  original: [100.  nan   0.  nan 150.]
  filled  : [100. 125.   0. 125. 150.]
  -> in_stock==0 is checked FIRST so a missing+out-of-stock item becomes 0, not the mean

--- 6: mini bar report ---
  [! ]  30/200  15.0%  [####..........................]
  -> 15% is in the 5-20% band: impute, d

---
# ⚠️ Common Mistakes

**Mistake 1 — Testing for NaN with `arr == np.nan`.**
This ALWAYS returns `False`, even where a value actually is NaN — by definition, `NaN` is not equal to anything, including itself.  
이 방식은 실제로 값이 NaN이어도 항상 `False`를 반환합니다 — 정의상 `NaN`은 자기 자신을 포함해 그 무엇과도 같지 않습니다.  
✅ **Fix:** Always use `np.isnan(arr)` to detect NaN — never `==`.  
✅ **해결법:** NaN을 확인할 때는 항상 `np.isnan(arr)`을 사용하세요 — `==`는 절대 안 됩니다.

**Mistake 2 — Calling `np.isnan()` directly on an `object`-dtype array.**  
If a column contains mixed types or came from Pandas as `object` dtype, `np.isnan(arr)` raises a `TypeError` instead of working.  
컬럼에 타입이 섞여 있거나 Pandas에서 `object` dtype으로 들어왔다면, `np.isnan(arr)`은 작동하지 않고 `TypeError`를 발생시킵니다.  
✅ **Fix:** Convert with `.astype(float)` first: `np.isnan(arr.astype(float))`.  
✅ **해결법:** 먼저 `.astype(float)`로 변환하세요: `np.isnan(arr.astype(float))`.

**Mistake 3 — Using `np.mean()` instead of `np.nanmean()` on data with gaps.**
`np.mean()` returns `nan` if even one element is `NaN` — the missing value silently contaminates the whole calculation.  
`np.mean()`은 원소가 하나라도 `NaN`이면 `nan`을 반환합니다 — 결측값 하나가 전체 계산을 조용히 오염시킵니다.  
✅ **Fix:** Use `np.nanmean()` / `np.nanmedian()` / `np.nansum()` whenever `NaN` might be present.  
✅ **해결법:** `NaN`이 있을 수 있다면 항상 `np.nanmean()` / `np.nanmedian()` / `np.nansum()`을 사용하세요.

**Mistake 4 — Applying `dropna()` wholesale without checking the missing rate first.**
On data with a high missing rate, an unqualified `dropna()` can silently discard the majority of rows — and if the missingness follows a pattern (one channel's data failed to collect entirely, say), dropping doesn't just lose data, it introduces bias.  
결측률이 높은 데이터에서 조건 없는 `dropna()`는 대부분의 행을 조용히 버릴 수 있습니다 — 결측이 특정 패턴(한 채널의 데이터 수집이 통째로 실패한 경우 등)을 따른다면, 제거는 단순한 데이터 손실이 아니라 편향을 만듭니다.  
✅ **Fix:** Check `.isna().mean()` per column BEFORE dropping, and use `dropna(subset=[...])` to target only the columns that truly matter.  
✅ **해결법:** 제거하기 전에 컬럼별 `.isna().mean()`을 먼저 확인하고, 정말 중요한 컬럼만 `dropna(subset=[...])`으로 지정해서 제거하세요.

**Mistake 5 — Imputing with the mean before handling outliers.**
If the data has outliers, the mean you compute for imputation is ITSELF already distorted by those outliers — you'd be filling gaps with a skewed number.  
데이터에 이상치가 있다면, 대체에 사용할 평균 자체가 이미 그 이상치로 왜곡되어 있습니다 — 왜곡된 숫자로 빈칸을 채우게 됩니다.  
✅ **Fix:** Handle outliers (clip or remove) first, then impute — or use the median, which resists outliers on its own.  
✅ **해결법:** 이상치를 먼저 처리(clip 또는 제거)한 후 대체하거나, 이상치에 강한 중앙값을 사용하세요.

**Mistake 6 — Using `fillna(method='ffill')`.**
This `method=` parameter has been REMOVED in recent pandas versions and raises a `TypeError` — it's not just deprecated, it no longer works at all.  
이 `method=` 파라미터는 최신 pandas 버전에서 완전히 제거되어 `TypeError`를 발생시킵니다 — 단순히 deprecated된 게 아니라 아예 작동하지 않습니다.  
✅ **Fix:** Use `.ffill()` / `.bfill()` directly instead of `fillna(method='ffill'/'bfill')`.  
✅ **해결법:** `fillna(method='ffill'/'bfill')` 대신 `.ffill()` / `.bfill()`을 직접 사용하세요.

**Mistake 7 — Interpolating a column that isn't actually ordered.**
`interpolate()` assumes neighboring rows are meaningfully related (like consecutive days) — running it on something like customer ID order produces values with no real meaning.  
`interpolate()`는 인접한 행들이 의미 있게 연결되어 있다고 가정합니다(연속된 날짜처럼) — 고객 ID 순서 같은 데이터에 적용하면 실제로 아무 의미 없는 값이 나옵니다.  
✅ **Fix:** Only interpolate columns where row order genuinely represents a sequence (time, distance, etc.).  
✅ **해결법:** 행 순서가 실제로 어떤 시퀀스(시간, 거리 등)를 의미하는 컬럼에만 보간을 적용하세요.

**Mistake 8 — Nesting `np.where()` 3+ levels deep for conditional replacement.**
Deeply nested `np.where()` calls become hard to read and easy to get the condition order wrong in.  
깊게 중첩된 `np.where()` 호출은 읽기 어렵고 조건 순서를 헷갈리기 쉽습니다.  
✅ **Fix:** Switch to `np.select(conditions, choices, default=...)` once you have more than 2-3 rules — it lists every condition and result as a flat, readable pair.  
✅ **해결법:** 규칙이 2~3개를 넘어가면 `np.select(conditions, choices, default=...)`로 바꾸세요 — 모든 조건과 결과를 평평하고 읽기 쉬운 쌍으로 나열합니다.

---
# 💡 Tips
Useful tips or shortcuts

- Run a missing-value check (`.isna().sum()`) as the very FIRST thing you do after loading any new dataset — before any other analysis.  
  새 데이터셋을 불러온 직후 가장 먼저 결측치 확인(`.isna().sum()`)을 하세요 — 다른 어떤 분석보다도 먼저.
- The rough rule of thumb from this section: <5% → drop is fine, 5-20% → impute, >20% → seriously consider dropping the whole column instead.  
  이 섹션의 대략적인 기준: 5% 미만 → 제거해도 무방, 5~20% → 대체, 20% 초과 → 컬럼 전체 제거를 진지하게 고려.
- Always log the before/after missing-value counts whenever you clean data — it's the first thing anyone auditing your pipeline will ask for.  
  데이터를 정제할 때는 항상 전후 결측치 개수를 기록하세요 — 파이프라인을 감사할 때 누구나 가장 먼저 물어볼 내용입니다.

---
# 🔗 Related Concepts

```
Section 11 — Business KPI Calculations
   (mean, median, std, growth/change rate, CVR, retention, CAC, LTV, ROAS, NPS)
        ↓
🔵 Section 12 — Missing Value Handling   ← you are here (final section)
   (detect, drop, impute, interpolate, conditional replace, report, pipeline)
        ↓
Pandas → SQL → Tableau
```

*How is today's topic connected to other concepts?*

This closes the loop on everything before it: `np.isnan()`/`np.isfinite()` (Section 9), `np.where()` (Section 5), `.reshape(-1,1)` Broadcasting (Section 7), `axis` direction (Section 8), and the `calculate_all_kpis()`-style pipeline function (Section 11) all reappear here as the toolkit for one of the most common real tasks in BA/DA work. From here, the natural next step — outside this NumPy series — is Pandas, SQL, and Tableau, where the same detect→clean→verify discipline applies to messier, larger, and more varied real-world data.

이 섹션은 지금까지의 모든 것과 이어집니다: `np.isnan()`/`np.isfinite()`(섹션 9), `np.where()`(섹션 5), `.reshape(-1,1)` Broadcasting(섹션 7), `axis` 방향(섹션 8), `calculate_all_kpis()` 스타일의 파이프라인 함수(섹션 11)가 전부 BA/DA 실무에서 가장 흔한 작업 중 하나를 위한 도구로 다시 등장합니다. 여기서부터 이 NumPy 시리즈 밖의 자연스러운 다음 단계는 Pandas, SQL, Tableau이며, 같은 탐지→정제→검증 원칙이 더 지저분하고 크고 다양한 실제 데이터에도 그대로 적용됩니다.

---
# 💼 Business Example — The Capstone: `missing_value_pipeline()`
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
Your team receives a new data export every month, and it always has some missing values. You need ONE function that: reports the missing rate per column, drops any column that's mostly empty, fills numeric columns (using group-aware statistics where possible), fills categorical columns with the mode, and verifies zero missing values remain.  
팀은 매달 새로운 데이터를 받는데, 항상 결측치가 조금씩 있습니다. 다음을 하는 함수 하나가 필요합니다: 컬럼별 결측률 리포트, 대부분 비어있는 컬럼 제거, 수치형 컬럼 대체(가능하면 그룹 인식 통계 사용), 범주형 컬럼은 최빈값 대체, 결측치가 0개 남았는지 검증.

**To-do / 할 일:**
- [x] Report the missing rate for every column.  
      모든 컬럼의 결측률을 리포트합니다.
- [x] Drop columns above a missing-rate threshold.  
      결측률이 기준을 넘는 컬럼을 제거합니다.
- [x] Impute numeric columns (group-aware if a group column is given).  
      수치형 컬럼을 대체합니다(그룹 컬럼이 주어지면 그룹별로).
- [x] Impute categorical columns with the mode, then verify.  
      범주형 컬럼은 최빈값으로 대체한 뒤 검증합니다.

In [10]:
import numpy as np
import pandas as pd

def missing_value_pipeline(df, drop_threshold=0.5, num_strategy="median", group_col=None):
    """
    End-to-end missing-value pipeline.
    Steps: 1) measure missing rate  2) drop columns above drop_threshold
           3) impute numeric columns (group-aware if group_col given)
           4) impute categorical columns with the mode  5) verify
    """
    print("=" * 55)
    print("   Missing-Value Pipeline")
    print("=" * 55)
    df_out = df.copy()
    n_orig = len(df_out)

    # Step 1: missing rate per column
    nan_pct = df_out.isna().mean()
    print("\n[Step 1] Missing rate per column:")
    for col, pct in nan_pct.items():
        print(f"  {col:>12}: {pct*100:.1f}%")

    # Step 2: drop columns above threshold
    drop_cols = nan_pct[nan_pct > drop_threshold].index.tolist()
    if drop_cols:
        df_out = df_out.drop(columns=drop_cols)
        print(f"\n[Step 2] Dropped columns (>{drop_threshold*100:.0f}% missing): {drop_cols}")
    else:
        print("\n[Step 2] No columns exceeded the drop threshold")

    # Step 3: numeric imputation
    num_cols = df_out.select_dtypes(include=np.number).columns.tolist()
    print(f"\n[Step 3] Imputing numeric columns (strategy: {num_strategy})")
    for col in num_cols:
        n_nan = df_out[col].isna().sum()
        if n_nan == 0:
            continue
        if group_col and group_col in df_out.columns:
            fill = df_out.groupby(group_col)[col].transform(num_strategy)
            df_out[col] = df_out[col].fillna(fill)
            arr = df_out[col].to_numpy().astype(float)
            if np.any(np.isnan(arr)):
                fallback = np.nanmedian(arr) if num_strategy == "median" else np.nanmean(arr)
                df_out[col] = df_out[col].fillna(fallback)
        else:
            arr = df_out[col].to_numpy().astype(float)
            fallback = np.nanmedian(arr) if num_strategy == "median" else np.nanmean(arr)
            df_out[col] = df_out[col].fillna(fallback)
        print(f"  {col:>12}: {n_nan} NaN -> imputed")

    # Step 4: categorical imputation
    cat_cols = df_out.select_dtypes(include=object).columns.tolist()
    print("\n[Step 4] Imputing categorical columns with the mode")
    for col in cat_cols:
        n_nan = df_out[col].isna().sum()
        if n_nan == 0:
            continue
        mode = df_out[col].mode()[0]
        df_out[col] = df_out[col].fillna(mode)
        print(f"  {col:>12}: {n_nan} NaN -> '{mode}' (mode)")

    # Step 5: verify
    remaining_nan = df_out.isna().sum().sum()
    print("\n[Step 5] Verification")
    print(f"  Before: {n_orig} rows / {df.isna().sum().sum()} NaN")
    print(f"  After:  {len(df_out)} rows / {remaining_nan} NaN")
    print("  All missing values handled" if remaining_nan == 0
          else f"  {remaining_nan} still missing -- manual check needed")
    print("=" * 55)

    return df_out


# ── Demo: a realistic monthly data export with several missing patterns ──
np.random.seed(42)
n = 500
df_raw = pd.DataFrame({
    "channel": np.where(np.random.rand(n) < 0.06, None,
                         np.random.choice(["search", "sns", "email"], n)),
    "revenue": np.where(np.random.rand(n) < 0.08, np.nan,
                         np.random.randint(10000, 500000, n).astype(float)),
    "visits": np.where(np.random.rand(n) < 0.05, np.nan,
                        np.random.randint(100, 5000, n).astype(float)),
    "cvr": np.where(np.random.rand(n) < 0.10, np.nan,
                     np.random.uniform(0.01, 0.1, n)),
    "useless": np.full(n, np.nan),   # entirely empty column -- should be dropped
})

df_clean = missing_value_pipeline(
    df_raw, drop_threshold=0.5, num_strategy="median", group_col="channel"
)

   Missing-Value Pipeline

[Step 1] Missing rate per column:
       channel: 7.2%
       revenue: 9.4%
        visits: 4.0%
           cvr: 11.2%
       useless: 100.0%

[Step 2] Dropped columns (>50% missing): ['useless']

[Step 3] Imputing numeric columns (strategy: median)
       revenue: 47 NaN -> imputed
        visits: 20 NaN -> imputed
           cvr: 56 NaN -> imputed

[Step 4] Imputing categorical columns with the mode
       channel: 36 NaN -> 'sns' (mode)

[Step 5] Verification
  Before: 500 rows / 659 NaN
  After:  500 rows / 0 NaN
  All missing values handled


/var/folders/9d/hwl9yp1n3rv1y9j90spl8rmh0000gn/T/ipykernel_11082/3705468765.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_out.select_dtypes(include=object).columns.tolist()


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

Missing value handling follows one consistent workflow: detect (`np.isnan()`, `np.isfinite()`, per-column and per-row counts), understand why the data is missing, choose a strategy — drop when missing data is rare and random, impute with a mean/median/mode when it's moderate, interpolate when the data is ordered/time-series, or replace conditionally when the cause is a known business rule — and verify nothing was missed. A text-based report makes the missing pattern visible without needing a plotting library, and everything comes together in a single reusable pipeline function that reports, drops, imputes, and verifies in one call. This closes out the entire NumPy series: the axis logic, Broadcasting, `np.where()` guards, and reusable-function habits built across all twelve sections all converge here, in the task every real dataset eventually requires.

결측치 처리는 하나의 일관된 워크플로우를 따릅니다: 탐지(`np.isnan()`, `np.isfinite()`, 컬럼별·행별 개수), 결측 원인 파악, 전략 선택(결측이 드물고 무작위면 제거, 중간 정도면 평균·중앙값·최빈값 대체, 순서가 있는 시계열이면 보간, 원인이 명확한 비즈니스 규칙이면 조건부 대체), 그리고 빠진 것이 없는지 검증. 텍스트 기반 리포트는 시각화 라이브러리 없이도 결측 패턴을 눈에 보이게 하며, 이 모든 것이 리포트·제거·대체·검증을 한 번에 처리하는 하나의 재사용 파이프라인 함수로 합쳐집니다. 이것으로 전체 NumPy 시리즈가 마무리됩니다: 12개 섹션에 걸쳐 쌓아온 axis 논리, Broadcasting, `np.where()` 방어, 재사용 함수 습관이 모두 여기, 모든 실제 데이터셋이 결국 마주하게 되는 이 작업에서 합쳐집니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Missing value handling is detect → understand → choose a strategy (drop/impute/interpolate/conditional) → verify — and it's the fitting final skill of this series, since it's the one task nearly every real dataset requires before any of the other eleven sections' tools can be trusted.

> 결측치 처리는 탐지 → 원인 파악 → 전략 선택(제거/대체/보간/조건부) → 검증이며, 다른 11개 섹션의 도구를 신뢰하기 전에 거의 모든 실제 데이터셋이 필요로 하는 작업이라는 점에서 이 시리즈의 마지막 스킬로 걸맞습니다.

---
# ❓ Review Questions

**Q1.** Why does `np.isnan()` raise a `TypeError` on some arrays, and how do you fix it?
`np.isnan()`은 왜 일부 배열에서 `TypeError`를 발생시키며, 어떻게 해결하나요?

**Q2.** What's the rough missing-rate threshold this section uses to decide between dropping and imputing, and why does a >20% column deserve extra scrutiny?
이 섹션에서 제거와 대체를 가르는 대략적인 결측률 기준은 무엇이며, 20%를 넘는 컬럼은 왜 더 신중하게 봐야 하나요?

**Q3.** Why is interpolation appropriate for a daily DAU series but not for a column ordered by customer ID?
보간이 일별 DAU 시계열에는 적합하지만 고객 ID 순서로 정렬된 컬럼에는 왜 부적합한가요?

**Q4.** In the conditional-replacement pattern from Example 5, why does the outer `np.where()` check `visitors == 0` BEFORE the inner one checks `np.isnan(revenue)`?
Example 5의 조건부 대체 패턴에서, 바깥쪽 `np.where()`가 안쪽에서 `np.isnan(revenue)`를 확인하기 전에 왜 `visitors == 0`을 먼저 확인하나요?

**Q5.** What are the five steps of `missing_value_pipeline()`, in order?
`missing_value_pipeline()`의 5단계는 순서대로 무엇인가요?

---
*📅 Try answering these again in a few days.*

---
### 🎉 Series complete / 시리즈 완료
This is the 12th and final section of the NumPy series (13 notebooks: Sections 1-10, plus Section 11 in two parts). Together they cover array fundamentals, indexing and manipulation, vectorized computation, the full BA/DA KPI toolkit, and the data-cleaning skills that tie it all together.

이것으로 NumPy 시리즈의 12번째이자 마지막 섹션이 끝났습니다(총 13개 노트북: 섹션 1~10 + 2부로 나뉜 섹션 11). 배열 기초, 인덱싱과 조작, 벡터화 연산, BA/DA KPI 전체 도구, 그리고 이 모든 것을 하나로 묶는 데이터 정제 스킬까지 다뤘습니다.